In [0]:
%run ./save_utils

In [0]:
import json
import pyspark.sql.functions as F

def load_json_to_bronze(source_path, target_table):
    df = (spark.read
            .option("multiline", "true")
            .json(source_path)
            .select("*", "_metadata.file_path")
            .withColumnRenamed("file_path", "_source_file")
            .withColumn("_ingestion_timestamp", F.current_timestamp())
            # .withColumn("_source_file", F.input_file_name())
        )
    df.write.format("delta").mode("append").saveAsTable(target_table)


def load_data_to_bronze(endpoint_key):
    entity = endpoint_key.value
    target_dir = f"{PROFILE.get(ProfileKeys.LANDING_ROOT)}/{entity}"
    full_save_path = f"{target_dir}/*.json"
    target_table = PROFILE.get(ProfileKeys.BRONZE_ROOT).format(entity_name= entity)
    raw_df = (spark.read
          .option("multiline", "true")
        #   .option("inferSchema", "true")
          .json(full_save_path)
        )
    display(raw_df)
    root_col = raw_df.columns[0]
    df = raw_df.select(f"{root_col}.*")
    display(df)
    
                       
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(target_table)
    print(f"Data loaded to {target_table} from {full_save_path}")

def load_countries_to_bronze(endpoint_key):
  entity = endpoint_key.value
  target_dir = f"{PROFILE.get(ProfileKeys.LANDING_ROOT)}/{entity}"
  full_source_path = f"{target_dir}/*.json"
  target_table = PROFILE.get(ProfileKeys.BRONZE_ROOT).format(entity_name= entity)

  raw_df = (spark.read.option("multiline", "true").json(full_source_path))
  
  # CountryResource -> Countries -> Country
  df_exploded = (raw_df.select(F.explode("CountryResource.Countries.Country").alias("country_data")))

  df_final = (df_exploded.select("country_data.*")
              # .withColumn("_source_file", F.input_file_name())
              # .withColumn("_ingestion_timestamp", F.current_timestamp())
              )

  df_final.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(target_table)
  print(f"Таблица {target_table} создана. Загружено строк: {df_final.count()}")